# 02 — EDA

Load cleaned datasets and perform:
- Per-column quality audits (missing values, unique counts, type checks)
- Gene ID overlap across transcriptomics datasets
- Cell line ID overlap across all datasets
- Summary statistics and key findings

In [ ]:
import sys, os
from pathlib import Path

# Resolve repo root: walk up from this file's location until we find src/scripts
_here = Path(os.path.abspath("__file__")).resolve() if "__file__" in dir() else Path.cwd()
_repo_root = next(
    (p for p in [_here, *_here.parents] if (p / "src" / "scripts").exists()),
    Path.cwd()
)
sys.path.insert(0, str(_repo_root / "src" / "scripts"))

import re
import pandas as pd
from data_utils import load_clean_parquets, preview

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## Load Cleaned Data

In [ ]:
tables = load_clean_parquets()
hpa_rna = tables["hpa_rna"]
depmap_expr = tables["depmap_expr"]
geo_expr = tables["geo_expr"]
proteomics = tables["proteomics"]
fusions = tables["fusions"]
mutations = tables["mutations"]
cellosaurus = tables["cellosaurus"]
depmap_profiles = tables["depmap_profiles"]
sample_info = tables["sample_info"]
geo_info = tables["geo_info"]
hpa_desc = tables["hpa_desc"]
metabolomics = tables["metabolomics"]
mirna = tables["mirna"]
signatures = tables["signatures"]

## Per-Column Quality Audit

In [ ]:
def audit(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Return a per-column quality summary (missing %, dtype, unique count, sample values)."""
    n = len(df)
    rows = []
    for col in df.columns:
        s = df[col]
        n_miss = s.isna().sum()
        n_uniq = s.nunique(dropna=True)
        sample = s.dropna().astype(str).head(3).tolist()
        rows.append({
            "column":        col,
            "dtype":         str(s.dtype),
            "missing_%":     round(n_miss / n * 100, 1) if n else 0,
            "unique_values": n_uniq,
            "sample":        ", ".join(sample),
        })
    result = pd.DataFrame(rows)
    print(f"\n{'='*60}")
    print(f" {name}  —  {df.shape[0]:,} rows x {df.shape[1]:,} cols")
    print(f"{'='*60}")
    display(result)
    return result

In [ ]:
audit(hpa_rna, "1. HPA RNA")

In [ ]:
audit(depmap_expr.iloc[:, :20], "2. DepMap Expression — first 20 gene cols")

In [ ]:
audit(geo_expr.iloc[:, :15], "3. GEO Expression — first 15 cols")

In [ ]:
audit(proteomics.iloc[:, :15], "4. Proteomics — first 15 cols")

In [ ]:
audit(fusions, "5. Fusions")

In [ ]:
mut_str_cols = mutations.select_dtypes(include="object").columns.tolist()
audit(mutations[mut_str_cols[:15]], "6. Mutations — string cols only")

In [ ]:
for name, df in [
    ("7. Cellosaurus",       cellosaurus),
    ("8. DepMap Profiles",   depmap_profiles),
    ("9. Sample Info",       sample_info),
    ("10. GEO Info",         geo_info),
    ("11. HPA Descriptions", hpa_desc),
    ("14. Global Signatures",signatures),
]:
    audit(df, name)

In [ ]:
audit(metabolomics.iloc[:, :15], "12. Metabolomics — first 15 cols")
audit(mirna.iloc[:, :10],        "13. miRNA — first 10 cols")

## Gene ID Overlap Across Transcriptomics Datasets

In [ ]:
ens_re = re.compile(r"ensg\d+", re.IGNORECASE)

# Extract ENSG IDs from each dataset
hpa_ensg    = set(hpa_rna["gene"].dropna())
geo_ensg    = set(geo_expr["gene"].dropna())
depmap_ensg = set(c for c in depmap_expr.columns if ens_re.match(str(c)))

print(f"HPA RNA genes:    {len(hpa_ensg):>7,}")
print(f"GEO Expr genes:   {len(geo_ensg):>7,}")
print(f"DepMap genes:     {len(depmap_ensg):>7,}")
print()
print(f"HPA ∩ GEO:        {len(hpa_ensg & geo_ensg):>7,}")
print(f"HPA ∩ DepMap:     {len(hpa_ensg & depmap_ensg):>7,}")
print(f"GEO ∩ DepMap:     {len(geo_ensg & depmap_ensg):>7,}")
print(f"All three:        {len(hpa_ensg & geo_ensg & depmap_ensg):>7,}")

## Cell Line ID Overlap Across Datasets

In [ ]:
# ACH- IDs
depmap_achs    = set(sample_info["depmap_id"].dropna())
profiles_achs  = set(depmap_profiles["modelid"].dropna())
metabol_achs   = set(metabolomics["depmap_id"].dropna())

print(f"Sample Info ACH IDs:    {len(depmap_achs):>5,}")
print(f"DepMap Profiles ACH IDs:{len(profiles_achs):>5,}")
print(f"Metabolomics ACH IDs:   {len(metabol_achs):>5,}")
print(f"Overlap (all 3):        {len(depmap_achs & profiles_achs & metabol_achs):>5,}")
print()

# HPA cell line names vs Cellosaurus
hpa_names = set(hpa_rna["cell line"].dropna()) if "cell line" in hpa_rna.columns else set()
cello_names = set(cellosaurus["cellosaurus_cell_line_name"].dropna())
print(f"HPA cell lines:         {len(hpa_names):>5,}")
print(f"Cellosaurus entries:    {len(cello_names):>5,}")
print(f"HPA names in Cello:     {len(hpa_names & cello_names):>5,}")

## Missing Data Heatmap Summary

In [ ]:
summary_rows = []
for name, df in tables.items():
    missing_pct = df.isnull().sum().sum() / df.size * 100 if df.size else 0
    cols_with_missing = (df.isnull().sum() > 0).sum()
    summary_rows.append({
        "Dataset": name,
        "Rows": f"{df.shape[0]:,}",
        "Cols": df.shape[1],
        "Missing %": f"{missing_pct:.1f}%",
        "Cols with nulls": cols_with_missing,
    })

pd.DataFrame(summary_rows)